In [1]:

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc


# -------------------------------------------------
# Device selection (ROCm or CPU)
# -------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc


# -------------------------------------------------
# Device selection (ROCm or CPU)
# -------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
! pip install kagglehub

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mrwellsdavid/unsw-nb15")

print("Path to dataset files:", path)

/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /root/.cache/kagglehub/datasets/mrwellsdavid/unsw-nb15/versions/1


In [6]:
train_dataset = "UNSW_NB15_training-set.csv"


dat = pd.read_csv(path+"/"+train_dataset)
dat = pd.DataFrame(dat)

In [8]:
dat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82332 entries, 0 to 82331
Data columns (total 45 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 82332 non-null  int64  
 1   dur                82332 non-null  float64
 2   proto              82332 non-null  object 
 3   service            82332 non-null  object 
 4   state              82332 non-null  object 
 5   spkts              82332 non-null  int64  
 6   dpkts              82332 non-null  int64  
 7   sbytes             82332 non-null  int64  
 8   dbytes             82332 non-null  int64  
 9   rate               82332 non-null  float64
 10  sttl               82332 non-null  int64  
 11  dttl               82332 non-null  int64  
 12  sload              82332 non-null  float64
 13  dload              82332 non-null  float64
 14  sloss              82332 non-null  int64  
 15  dloss              82332 non-null  int64  
 16  sinpkt             823

In [ ]:
import pandas as pd
import numpy as np
import os

# Path to the dataset file
train_folder = "/root/.cache/kagglehub/datasets/patrickfleith/nasa-anomaly-detection-dataset-smap-msl/versions/1/data/data/train"
train_data = {}

for fname in os.listdir(train_folder):
    if fname.endswith(".npy"):
        subsystem_name = fname.replace(".npy", "")
        train_data[subsystem_name] = np.load(os.path.join(train_folder, fname))

# Example: access one subsystem
print(train_data["C-1"].shape)  # shape of that subsystem’s data
print(train_data["A-1"].shape)  # shape of that subsystem’s data


In [ ]:
train_data["T-12"]

# OLD

In [ ]:
val = df[df["Class"] == 1]
ones = val.shape[0]
val = df[df["Class"] == 0]
zeros = val.shape[0]

print(f"Percent frauds: {(ones)/(zeros+ones)*100}%")

In [ ]:
# -------------------------------------------------
# Prepare targets
# -------------------------------------------------

dat = df.drop(columns=["Class","Time"])
target = df["Class"].values
target

In [ ]:
# -------------------------------------------------
# Prepare features
# -------------------------------------------------

max = dat["Amount"].max()
min = dat["Amount"].min()
dat["Amount"] = (dat["Amount"] - min) / (max - min)
dat.head()

In [ ]:
# -------------------------------------------------
# Final feature and target arrays
# -------------------------------------------------

X = dat.values
y = target
X,y

In [ ]:
# -------------------------------------------------
# Sanity check
# -------------------------------------------------

print(dat[["V1", "V21", "Amount"]].describe())
print(dat.isna().sum())

In [ ]:
# -------------------------------------------------
# Train-test split
# -------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# -------------------------------------------------
# Convert to PyTorch tensors and move to device
# -------------------------------------------------
X_train_gpu = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_gpu  = torch.tensor(X_test,  dtype=torch.float32).to(device)

y_train_gpu = torch.tensor(y_train, dtype=torch.long).to(device)
y_test_gpu  = torch.tensor(y_test,  dtype=torch.long).to(device)
y_train_gpu = y_train_gpu.float().unsqueeze(1)
y_test_gpu  = y_test_gpu.float().unsqueeze(1)


In [ ]:
#-------------------------------------------------
# Hyperparameters
#-------------------------------------------------
epochs = 30
batch_size = 64
lr = 0.001
input_size = X_train_gpu.shape[1]

# -------------------------------------------------
# Define a simple MLP model
# -------------------------------------------------
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        return self.net(x)

model = MLP().to(device)

# -------------------------------------------------
# Loss and optimizer
# -------------------------------------------------

num_neg = (y_train == 0).sum()
num_pos = (y_train == 1).sum()
pos_weight = torch.tensor(num_neg / num_pos).to(device)  # ~578 for this dataset

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight) 

# Alternative without pos_weight. The result is much worse
#criterion = nn.BCEWithLogitsLoss()


optimizer = optim.Adam(model.parameters(), lr=lr)

# -------------------------------------------------
# Training loop
# -------------------------------------------------


dataset = torch.utils.data.TensorDataset(X_train_gpu, y_train_gpu)
loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        preds = model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(loader):.4f}")

#


In [ ]:
# Release GPU memory
del X_train_gpu
del y_train_gpu
X_test_gpu  = torch.tensor(X_test,  dtype=torch.float32).to(device)

y_test_gpu  = torch.tensor(y_test,  dtype=torch.long).to(device)
y_test_gpu  = y_test_gpu.float().unsqueeze(1)

# Evaluation

In [ ]:

model.eval()
all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():
    for i in range(0, len(X_test_gpu), batch_size):
        xb = X_test_gpu[i:i+batch_size]
        yb = y_test_gpu[i:i+batch_size]

        logits = model(xb)                  # raw logits
        probs = torch.sigmoid(logits)       # probabilities [0-1]
        preds = (probs > 0.5).float()       # binary predictions (threshold 0.5)

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())

all_probs = np.array(all_probs).squeeze()
all_preds = np.array(all_preds).squeeze()
all_labels = np.array(all_labels).squeeze()

# Basic report
print(classification_report(all_labels, all_preds, target_names=['Non-Fraud', 'Fraud']))

# AUC scores (very important for imbalance)
print("ROC AUC:", roc_auc_score(all_labels, all_probs))
precision, recall, _ = precision_recall_curve(all_labels, all_probs)
print("PR AUC:", auc(recall, precision))